# 📈 Semana 3 · Unidad 1 — Introducción al Análisis de Algoritmos
Bienvenidos a la tercera semana del curso de Algoritmos y Estructuras de Datos. 
En esta clase traduciremos todas las lecciones del **Análisis de Algoritmos I y II** al lenguaje de programación Python.

## 1. Introducción y Propiedades de los Algoritmos
Antes de analizar cuán bueno o malo es un algoritmo, debemos comprender sus propiedades principales.
Un **algoritmo** debe ser:
- **Preciso**: pasos ordenados y sin ambigüedades.
- **Determinista**: produce el mismo resultado ante las mismas entradas.
- **Finito**: termina en un número finito de pasos.
- **Efectivo**: cada paso se puede llevar a cabo en tiempo finito.

## 🎯 Objetivos de Aprendizaje

Al finalizar esta sesión, el estudiante será capaz de:

1. **Comprender** las propiedades que debe cumplir un algoritmo: finitud, precisión y entradas y salidas bien definidas.
2. **Identificar** la diferencia entre el análisis empírico (medir tiempos) y el teórico (contar operaciones).
3. **Implementar** mediciones de tiempo reproducibles con `time` y `timeit`.
4. **Analizar** la complejidad espacial además de la temporal, y reconocer cuándo la memoria es el cuello de botella.
5. **Resolver** la recurrencia de la búsqueda binaria y justificar su costo $O(\log n)$.

## Verificación de Dependencias


In [ ]:
import math
import sys
import time
import unittest

def mostrar_barras(filas, unidad="ms", ancho=40, formato="8.2f", maximo=None):
    """
    Imprime un gráfico de barras horizontal en texto.

    Parámetros:
        filas (list[tuple[str, float]]): pares (etiqueta, valor).
        unidad (str): unidad que se muestra junto a cada valor.
        ancho (int): caracteres de la barra que representa a `maximo`.
        formato (str): formato del valor impreso, por ejemplo "8.2f" o "8,.0f".
        maximo (float | None): valor que ocupa todo el ancho; por defecto, el mayor
            de `filas`. Pasar el mismo valor a dos gráficos los deja en la misma escala.
    """
    maximo = maximo or max(valor for _, valor in filas) or 1
    margen = max(len(etiqueta) for etiqueta, _ in filas)
    for etiqueta, valor in filas:
        barra = "█" * round(valor / maximo * ancho) or ("▏" if valor > 0 else "")
        print(f"{etiqueta:>{margen}} │{barra:<{ancho}} {valor:{formato}} {unidad}")


print("🐍 Python", sys.version.split()[0], "| Todo listo para comenzar.")


## 2. Análisis Empírico de Algoritmos
El análisis empírico consiste en emplear el **método científico** para observar el comportamiento temporal de un programa en ejecución. Para esto, se miden los tiempos de ejecución al variar el tamaño de la entrada ($N$). Las gráficas obtenidas normalmente pueden confirmar las **Leyes de Potencia**.

Veamos un ejemplo de **Fuerza Bruta** clásico, el problema 3-Sum:

In [ ]:
def three_sum_brute(arr):
    """Algoritmo 3-Sum (Fuerza Bruta)
    Complejidad teórica: O(N^3)
    """
    n = len(arr)
    count = 0
    for i in range(n):
        for j in range(i+1, n):
            for k in range(j+1, n):
                if arr[i] + arr[j] + arr[k] == 0:
                    count += 1
    return count

# Experimento: Midiendo el tiempo de ejecución empírico
sizes = [25, 50, 100, 200]
times = []

for size in sizes:
    arr = list(range(-size//2, size//2))  # Arreglo de prueba
    
    start = time.perf_counter()   # reloj de alta resolución
    three_sum_brute(arr)
    end = time.perf_counter()
    
    times.append(end - start)

# Mostrar los resultados como gráfico de barras en texto
print("Tiempo de ejecución empírico de 3-Sum (fuerza bruta)\n")
mostrar_barras([(f"N={n}", t * 1000) for n, t in zip(sizes, times)])

# Ley de potencia: si T(N) ~ a·N³, al duplicar N el tiempo se multiplica por 2³ = 8
print()
for i in range(1, len(sizes)):
    razon = times[i] / times[i - 1] if times[i - 1] > 0 else float("inf")
    print(f"  N: {sizes[i-1]:>3} → {sizes[i]:>3} | tiempo ×{razon:.1f} (esperado ≈ ×8)")


> 🎙️ **[PAUSA PROFESOR]** Pregunta sugerida: *"Mido un algoritmo en mi notebook y tarda 2 segundos. ¿Qué puedo concluir sobre su complejidad?"* — Nada por sí solo: hay que medir varios tamaños y mirar cómo crece la curva, no un punto aislado.

## 3. Análisis Teórico y Notaciones Asintóticas
El análisis teórico busca encontrar un modelo matemático que describa el costo (típicamente el número de operaciones básicas o requerimientos de memoria) en función de $N$.
Se desprecian las constantes y términos menores mediante la regla de la tilde ($\sim$).

**Notación Asintótica** (ver el notebook de Introducción, sección 8):
- **Cota superior ($O$)**: $f(n) = O(g(n))$ significa que $f$ crece **a lo más** como $g$.
- **Cota inferior ($\Omega$)**: $f$ crece **al menos** como $g$.
- **Cota ajustada ($\Theta$)**: $f$ crece **exactamente** como $g$: es $O$ y $\Omega$ a la vez.

> ⚠️ $O$ **no** significa «peor caso» ni $\Omega$ «mejor caso». Los casos describen la entrada;
> las cotas describen el crecimiento. El peor caso de 3-Sum por fuerza bruta es $\Theta(N^3)$, y
> su mejor caso **también**: recorre todos los tríos sin importar los datos.

## 4. Complejidad Espacial
Aparte del tiempo, los recursos de ejecución impactan fuertemente en software real.
**Complejidad Espacial** es la cantidad de memoria extra que requiere un algoritmo en función de $N$.

- **Ordenamiento por Selección** -> El espacio adicional es $O(1)$, pues reorganiza *in situ*.
- **Fibonacci recursivo sin memoización** -> Su pila de ejecución llega a profundidad $O(N)$.

## 5. Búsqueda Binaria y Relaciones de Recurrencia
La **Búsqueda Binaria** es un algoritmo clásico del tipo *Divide y Vencerás* que opera en arreglos ordenados separando los elementos por la mitad.

La fórmula recurrente se escribe como:
$$ T(N) = T(N/2) + c $$

Usando métodos formales (como el **Método de Sustitución**, el **Árbol de Recursión** y el **Teorema Maestro**), comprobamos que el algoritmo divide su tamaño N tantas veces como se requiera hasta llegar al caso base. 

Matemáticamente, para $a=1$, $b=2$ y $f(n)=c$, tenemos $n^{\log_b a} = n^0 = 1$, que crece igual que $f(n)$. Ese es el **Caso 2** del Teorema Maestro, y la complejidad de tiempo resultante es logarítmica: $\Theta(\log N)$.

Veamos la implementación moderna de Búsqueda Binaria en Python:

In [ ]:
def busqueda_binaria(arr, target):
    lo = 0
    hi = len(arr) - 1
    
    while lo <= hi:
        mid = (lo + hi) // 2
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            lo = mid + 1      # Seguir iterando a la derecha
        else:
            hi = mid - 1      # Seguir iterando a la izquierda
            
    return -1 # Caso No Exitoso

# Demostración
a = [2, 5, 8, 12, 16, 23, 38, 56, 72, 91]
print("Busqueda exitosa para 23. Índice encontrado:", busqueda_binaria(a, 23))
print("Busqueda no exitosa para 34. Índice encontrado:", busqueda_binaria(a, 34))

# Ejercicios Prácticos


## 🧪 Ejercicio 1 ⭐: Búsqueda binaria instrumentada

**Descripción:** implementa la búsqueda binaria sobre un arreglo ordenado y devuelve
**además** cuántas comparaciones hizo. El conteo es lo que permite comprobar que el costo
es $O(\log n)$ y no algo peor.

**Entrada:** `arr` (list ordenada) y `objetivo`.
**Salida:** tupla `(indice, comparaciones)`; `indice` es `-1` si el objetivo no está.

**Ejemplo:**
```
Entrada: arr = [1, 3, 5, 7, 9], objetivo = 7
Salida:  (3, 2)
```

**Restricciones:** cuenta **una** comparación por cada vez que examinas `arr[medio]`.
**Complejidad esperada:** O(log n)

In [ ]:
def busqueda_binaria_contada(arr, objetivo):
    """
    Busca `objetivo` en `arr` (ordenado) contando las comparaciones.

    Parámetros:
        arr (list): arreglo ordenado ascendentemente
        objetivo: valor a buscar
    Retorna:
        tuple[int, int]: (índice o -1, número de comparaciones)
    """
    # Tu código aquí
    pass

In [ ]:
# Pruebas automáticas del ejercicio 1 (unittest)

class TestEjercicio1(unittest.TestCase):
    """busqueda_binaria_contada retorna (índice, comparaciones) dentro de la cota logarítmica."""

    CASOS = [
        ([1, 3, 5, 7, 9],          7,    3,   "el ejemplo del enunciado"),
        ([1, 3, 5, 7, 9],          1,    0,   "primer elemento"),
        ([1, 3, 5, 7, 9],          9,    4,   "último elemento"),
        ([1, 3, 5, 7, 9],          4,    -1,  "ausente, en un hueco"),
        ([],                       5,    -1,  "arreglo vacío"),
        ([42],                     42,   0,   "un solo elemento"),
        (list(range(0, 2000, 2)),  1998, 999, "n = 1000, elemento final"),
    ]

    def test_1_indice(self):
        """los 7 casos devuelven el índice correcto (o -1)"""
        for arr, objetivo, esperado, descripcion in self.CASOS:
            with self.subTest(descripcion):
                resultado = busqueda_binaria_contada(arr[:], objetivo)
                self.assertIsInstance(resultado, tuple, "debe retornar una tupla (índice, comparaciones)")
                self.assertEqual(resultado[0], esperado)

    def test_2_cota_logaritmica(self):
        """las comparaciones no superan log₂ n + 2"""
        for arr, objetivo, _, descripcion in self.CASOS:
            with self.subTest(descripcion):
                _, comparaciones = busqueda_binaria_contada(arr[:], objetivo)
                cota = int(math.log2(len(arr))) + 2 if arr else 1
                self.assertLessEqual(comparaciones, cota)

    def test_3_ejemplo(self):
        """[1, 3, 5, 7, 9] buscando 7 → (3, 2)"""
        self.assertEqual(busqueda_binaria_contada([1, 3, 5, 7, 9], 7), (3, 2))


resultado = unittest.main(argv=["ignorado", "TestEjercicio1"], exit=False, verbosity=2)
if resultado.result.wasSuccessful():
    print("\n🎉 Todos los casos pasaron")


In [ ]:
# ═══════════════════════════════════════════════════
# SOLUCIÓN — Descomenta para ver después de intentarlo
# ═══════════════════════════════════════════════════

# def busqueda_binaria_contada(arr, objetivo):
#     """Búsqueda binaria iterativa, contando cada examen de arr[medio]."""
#     lo, hi, comparaciones = 0, len(arr) - 1, 0
#     while lo <= hi:
#         medio = (lo + hi) // 2
#         comparaciones += 1              # una comparación por examen
#         if arr[medio] == objetivo:
#             return medio, comparaciones
#         if arr[medio] < objetivo:
#             lo = medio + 1              # descarto la mitad izquierda
#         else:
#             hi = medio - 1              # descarto la mitad derecha
#     return -1, comparaciones
#     # Complejidad: O(log n) temporal, O(1) espacial

## 🧪 Ejercicio 2 ⭐⭐: Estimar el exponente empíricamente

**Descripción:** dado un algoritmo, estima su exponente $k$ suponiendo que su costo es
$T(n) \approx c \cdot n^k$. Si al **duplicar** $n$ el costo se multiplica por $r$, entonces

$$k = \log_2 r$$

Es la técnica del *doubling ratio* de Sedgewick, y sirve para descubrir la complejidad de
un algoritmo sin leer su código.

**Entrada:**
- `medir` (callable): recibe `n` y devuelve un costo (operaciones o segundos)
- `n0` (int): tamaño inicial
- `repeticiones` (int): cuántas veces duplicar

**Salida:** el exponente estimado (float), promediando los $\log_2$ de las razones.

**Ejemplo:**
```
Entrada: medir = lambda n: n*n, n0 = 100, repeticiones = 4
Salida:  2.0      (aprox.)
```

**Restricciones:** redondea a 2 decimales al comparar; ignora las razones no positivas.
**Complejidad esperada:** O(repeticiones · costo de medir)

In [ ]:
def estimar_exponente(medir, n0=100, repeticiones=4):
    """
    Estima k suponiendo T(n) ~ c·n^k, duplicando n y mirando la razón de costos.

    Parámetros:
        medir (callable): función n -> costo
        n0 (int): tamaño inicial
        repeticiones (int): cuántas duplicaciones hacer
    Retorna:
        float: exponente estimado
    """
    # Tu código aquí
    pass

In [ ]:
# Pruebas automáticas del ejercicio 2 (unittest)

class TestEjercicio2(unittest.TestCase):
    """estimar_exponente recupera k cuando el costo es c·n^k."""

    def test_1_potencias(self):
        """funciones de exponente conocido"""
        casos = [
            (lambda n: n,      100, 5, 1.0, "lineal: k = 1"),
            (lambda n: n * n,  100, 5, 2.0, "cuadrático: k = 2"),
            (lambda n: n ** 3, 50,  4, 3.0, "cúbico: k = 3"),
            (lambda n: 1,      100, 4, 0.0, "constante: k = 0"),
        ]
        for medir, n0, repeticiones, esperado, descripcion in casos:
            with self.subTest(descripcion):
                self.assertAlmostEqual(estimar_exponente(medir, n0, repeticiones), esperado, delta=0.15)

    def test_2_n_log_n(self):
        """n·log n no es una potencia pura: k queda entre 1 y 1,35"""
        k = estimar_exponente(lambda n: n * math.log2(n), 100, 5)
        self.assertGreater(k, 1.0)
        self.assertLess(k, 1.35)

    def test_3_sobre_3sum(self):
        """sobre el conteo real de 3-Sum (tríos revisados) da k ≈ 3"""
        k = estimar_exponente(lambda n: n * (n - 1) * (n - 2) // 6, 50, 4)
        self.assertAlmostEqual(k, 3.0, delta=0.15)


resultado = unittest.main(argv=["ignorado", "TestEjercicio2"], exit=False, verbosity=2)
if resultado.result.wasSuccessful():
    print("\n🎉 Todos los casos pasaron")


In [ ]:
# ═══════════════════════════════════════════════════
# SOLUCIÓN — Descomenta para ver después de intentarlo
# ═══════════════════════════════════════════════════

# def estimar_exponente(medir, n0=100, repeticiones=4):
#     """Doubling ratio: si al duplicar n el costo se multiplica por r, k = log2(r)."""
#     import math
#     n = n0
#     anterior = medir(n)
#     exponentes = []
#     for _ in range(repeticiones):
#         n *= 2                          # Paso 1: duplicar el tamaño
#         actual = medir(n)
#         # Paso 2: la razón de costos; se ignoran las medidas no positivas
#         if anterior > 0 and actual > 0:
#             exponentes.append(math.log2(actual / anterior))
#         anterior = actual
#     # Paso 3: promediar las estimaciones
#     return sum(exponentes) / len(exponentes) if exponentes else 0.0
#     # Complejidad: O(repeticiones) llamadas a medir

## 🔬 Zona de Experimentación

Sugerencias:
- Usa `estimar_exponente` con una función que **mida el tiempo real** de `three_sum_brute`. ¿Qué tan cerca de 3 queda?
- Cuenta las comparaciones de `busqueda_binaria_contada` para n = 10³, 10⁶ y 10⁹ (sin crear el arreglo: usa `range`).
- Escribe la recurrencia de un algoritmo que divide el problema en **tres** partes y resuelve solo una. ¿Qué caso del Teorema Maestro es?


In [ ]:
# Espacio libre para experimentar


## 📚 Lecturas Recomendadas y Práctica

### Textos del curso

| Libro | Edición | Capítulo | Tema |
|-------|---------|----------|------|
| Cormen et al. (CLRS) — *Introduction to Algorithms* | 4ª ed. | Cap. 2 | Insertion sort y el análisis de algoritmos |
| Cormen et al. (CLRS) — *Introduction to Algorithms* | 4ª ed. | Cap. 4.3–4.5 | Resolución de recurrencias y teorema maestro |
| Miller & Ranum (M&R) — *Problem Solving with Algorithms and Data Structures Using Python* | 2011 | Cap. 2 | Análisis empírico con `timeit` |
| Goodrich, Tamassia & Goldwasser (GTG) — *Data Structures and Algorithms in Python* | 1ª ed. | Cap. 3 | Complejidad temporal y espacial |

### Recursos gratuitos en línea

- 📄 [timeit — documentación oficial](https://docs.python.org/es/3/library/timeit.html) — cómo medir sin engañarse.
- 🌐 [Big-O Cheat Sheet](https://www.bigocheatsheet.com/) — referencia rápida de costos.

### Práctica en Codeforces (soporta Python 3)

> 🔍 **Cómo filtrar:** ve a [codeforces.com/problemset](https://codeforces.com/problemset),
> escribe la etiqueta en **Tags** y ajusta **Rating**.

**Escala de dificultad orientativa para este curso:**

| Rating | Nivel | Descripción |
|--------|-------|-------------|
| 800 | ⭐ | Aplicación directa — la mayoría puede resolverlo |
| 1000–1200 | ⭐⭐ | Requiere una pequeña adaptación |
| 1300+ | ⭐⭐⭐ | Combina la idea con otra — desafío |

**Problemas recomendados para este tópico:**

| # | Problema | Rating | Por qué es útil |
|---|----------|--------|-----------------|
| 1 | [158A — Next Round](https://codeforces.com/problemset/problem/158/A) | ⭐ 800 | Un recorrido lineal: mide y compara con la predicción |
| 2 | [474B — Worms](https://codeforces.com/problemset/problem/474/B) | ⭐⭐ 1200 | Búsqueda binaria sobre sumas acumuladas: O(n) vs O(log n) se nota |
| 3 | [466C — Number of Ways](https://codeforces.com/problemset/problem/466/C) | ⭐⭐⭐ 1700 | La solución ingenua es O(n²); hay que llegar a O(n) |

⚠️ Los dos primeros son el **mínimo esperado**. Los demás son desafío opcional.